In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
import oracledb
import oci
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_oracledb.vectorstores import oraclevs
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from oci.generative_ai_inference import models
from langchain_community.embeddings import OCIGenAIEmbeddings

print("Successfully imported libraries and modules")

# Load DB credentials from .env. DB_PASSWORD is also the PEM/wallet password.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / ".env").exists() and Path("/Users/vavena/Documents/Playground/.env").exists():
    PROJECT_DIR = Path("/Users/vavena/Documents/Playground")

load_dotenv(PROJECT_DIR / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
missing_env = [name for name, value in {"DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}.items() if not value]
if missing_env:
    raise RuntimeError(f"Missing required value(s) in .env: {', '.join(missing_env)}")

dsn = """(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1522)(host=adb.us-phoenix-1.oraclecloud.com))(connect_data=(service_name=g10d1d163445a60_ragval_high.adb.oraclecloud.com))(security=(ssl_server_dn_match=yes)))"""


# Connect to the database
try:
    conn23c = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=dsn)
    print("Connection successful!")
except Exception as e:
    print("Connection failed!")
    print(type(e).__name__, e)
    raise

# Retrieval Step 1 - Build the llm, embed_model and prompt to query the document

COMPARTMENT_OCID = "ocid1.compartment.oc1..aaaaaaaaat66um3xnwruivicqzcwelw3owwbce2ykrp72snonmlomngdb4ya"

endpoint = "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com"

config = oci.config.from_file("~/.oci/config", "DEFAULT")

signer = oci.auth.signers.SecurityTokenSigner(
    token=open(config["security_token_file"], "r").read(),
    private_key=oci.signer.load_private_key_from_file(config["key_file"]),
)

client = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config={},
    signer=signer,
    service_endpoint=endpoint,
)


def ask_oci_openai(prompt_text: str) -> str:
    chat_detail = models.ChatDetails(
        compartment_id=COMPARTMENT_OCID,
        serving_mode=models.OnDemandServingMode(model_id="openai.gpt-5.5"),
        chat_request=models.GenericChatRequest(
            messages=[models.UserMessage(content=[models.TextContent(text=prompt_text)])],
            max_completion_tokens=1500,
        ),
    )
    result = client.chat(chat_detail).data
    return result.chat_response.choices[0].message.content[0].text

embed_model = OCIGenAIEmbeddings(
    model_id="openai.text-embedding-3-small",
    service_endpoint=endpoint,
    compartment_id=COMPARTMENT_OCID,
    auth_type="SECURITY_TOKEN",
    auth_profile="DEFAULT",
    auth_file_location="~/.oci/config",
)



# Set up the template for the questions and context, and instantiate the database retriever object
template = """Answer the question based only on the following context:
             {context} Question: {question} """
prompt = PromptTemplate.from_template(template)

# Retrieval Step 2 - Create retriever without ingesting documents again.

vs = OracleVS(
    embedding_function=embed_model,
    client=conn23c,
    table_name="MY_DEMO",
    distance_strategy=DistanceStrategy.DOT_PRODUCT
)

retriever = vs.as_retriever(search_type="similarity", search_kwargs={'k': 3})

user_question = "Tell us about Advanced Clustering"

context_docs = retriever.invoke(user_question)
context = "\n\n".join(doc.page_content for doc in context_docs)
response = ask_oci_openai(prompt.format(context=context, question=user_question))

print("User questions was ->", user_question)
print("Retrieved documents ->", len(context_docs))
print("LLM response is->", response)


/var/folders/kb/r11381mx63s1yqxzyk21tcdm0000gp/T/ipykernel_73654/242147566.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import DistanceStrategy


Successfully imported libraries and modules
Connection successful!


/Users/vavena/Documents/Playground/.venv-jupyter/lib/python3.14/site-packages/urllib3/poolmanager.py:329: FutureWarning: The 'strict' parameter is no longer needed on Python 3+. This will raise an error in urllib3 v3.0.
  warnings.warn(


User questions was -> Tell us about Advanced Clustering
Retrieved documents -> 3
LLM response is-> Advanced Clustering is an enterprise-specific clustering solution that uses data mining to create store groupings at different product levels. It helps identify patterns in data so businesses can create customer-centric, localized, and targeted clusters.

It can be used for:

- Localized or customer-centric assortments
- Pricing
- Forecasting, such as clustering stores with similar seasonal patterns
- Allocation, by grouping stores with similar selling patterns
- Assortment planning, replenishment, pricing, and promotion processes

Advanced Clustering uses a variety of inputs, including:

- Performance data, such as sales dollars, sales units, and gross profit
- Product attributes, such as brand, color, and size/fit
- Store attributes, such as climate, store format, size, and servicing distribution center
- Third-party demographic data, such as income, ethnicity, and population density
- 